In [ ]:
# Authenticate with GCP services and Gmail
# Use Buckets and call MCP server hosted on GCP
# Use Gmail tools so LLM can send emails.

In [ ]:
# 
# AUTHENTICATED ACCESS TO PRIVATE GCS BUCKETS
# 

import os
from pathlib import Path

# Change working directory to project root (parent of notebooks/)
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(project_root)
print(f"✓ Changed working directory to: {os.getcwd()}")

from dotenv import load_dotenv
from google.cloud import storage
from google.oauth2 import service_account

# Load .env file from project root
result = load_dotenv()
if result:
    print(f"✓ Loaded .env from: {Path('.env').absolute()}")
else:
    print(f"⚠ Warning: .env file not found at {Path('.env').absolute()}")

PROJECT_ID = os.environ["GOOGLE_CLOUD_PROJECT"] 
SERVICE_ACCOUNT_KEY_PATH = os.environ["GOOGLE_APPLICATION_CREDENTIALS"]
SECRETS_PATH = os.environ["SECRETS_PATH"]
BUCKET_NAME = "fionaa-customer-assets"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_KEY_PATH,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
client = storage.Client(project=PROJECT_ID, credentials=credentials)
bucket = client.bucket(BUCKET_NAME)


In [ ]:
# List all blobs/files in the bucket
blobs = list(bucket.list_blobs())

# Print the file names
print(f"Files in bucket '{bucket}':")
for blob in blobs:
    print(blob.name)

# Example: Read the contents of a file (e.g., PDF as bytes)
file_name = "data/5573 - Draft Accounts.pdf"
blob = bucket.blob(file_name)
file_bytes = blob.download_as_bytes()
print(f"Downloaded {file_name}, {len(file_bytes)} bytes")




In [ ]:
# This is for if we want to turn Gmail into a tool, not just ingesting emails.
# Might want to do this for the response agent when we need more details
# from the user such as missing information or clarification on the users request.

from langchain_google_community import GmailToolkit
from langchain_google_community.gmail.utils import (
    build_resource_service,
    get_gmail_credentials,
)

# Use the same scopes that were used when creating the token
# These must match the scopes in token.json
SCOPES = [
    'https://www.googleapis.com/auth/gmail.modify',
    'https://www.googleapis.com/auth/calendar'
]

credentials = get_gmail_credentials(
    token_file=SECRETS_PATH+"token.json",
    scopes=SCOPES,
    client_sercret_file=SECRETS_PATH+"/credentials.json"  # Note: setup script uses secrets.json
)
api_resource = build_resource_service(credentials=credentials)
toolkit = GmailToolkit(api_resource=api_resource)

In [ ]:
tools = toolkit.get_tools()
tools

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# Use LangGraph instead of create_agent (which has compatibility issues)
from typing import Literal

from langgraph.graph import END, START, MessagesState, StateGraph


# Define state for the agent
class AgentState(MessagesState):
    pass

# Get tools from toolkit and create a tools_by_name dict
tools = toolkit.get_tools()
tools_by_name = {tool.name: tool for tool in tools}

# Bind tools to LLM
llm_with_tools = llm.bind_tools(tools)

# Define LLM node
def llm_call(state: AgentState):
    """LLM decides whether to call a tool or not"""
    return {
        "messages": [
            llm_with_tools.invoke(state["messages"])
        ]
    }

# Define tool execution node
def tool_node(state: AgentState):
    """Execute tool calls"""
    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append({
            "role": "tool", 
            "content": str(observation), 
            "tool_call_id": tool_call["id"]
        })
    return {"messages": result}

# Conditional edge function
def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    """Route to tools if there are tool calls, otherwise end"""
    messages = state["messages"]
    last_message = messages[-1]
    if last_message.tool_calls:
        return "tools"
    return "__end__"

# Build the agent graph
agent_builder = StateGraph(AgentState)
agent_builder.add_node("llm", llm_call)
agent_builder.add_node("tools", tool_node)
agent_builder.add_edge(START, "llm")
agent_builder.add_conditional_edges(
    "llm",
    should_continue,
    {
        "tools": "tools",
        "__end__": END,
    },
)
agent_builder.add_edge("tools", "llm")

# Compile the agent
agent_executor = agent_builder.compile()

In [ ]:
from langchain_core.messages import HumanMessage

example_query = "Draft an email to fake@fake.com thanking them for coffee."

# Invoke the agent (LangGraph uses invoke, not stream with stream_mode)
result = agent_executor.invoke({
    "messages": [HumanMessage(content=example_query)]
})

# Print the final messages
for message in result["messages"]:
    print(f"{message.__class__.__name__}: {message.content}")
    if hasattr(message, 'tool_calls') and message.tool_calls:
        print(f"  Tool calls: {message.tool_calls}")

In [ ]:
example_query = "get the emails received in the past 24 hours."

# Invoke the agent (LangGraph uses invoke, not stream with stream_mode)
result = agent_executor.invoke({
    "messages": [HumanMessage(content=example_query)]
})

# Print the final messages
for message in result["messages"]:
    print(f"{message.__class__.__name__}: {message.content}")
    if hasattr(message, 'tool_calls') and message.tool_calls:
        print(f"  Tool calls: {message.tool_calls}")